# FloodShift 08B — Full Grounded Training Corpus

This notebook expands the grounded multimodal instruction corpus **before** variational-VLM training.

The key change is that the language model will no longer train on only 27 unique visual scenes.

Instead:

\[
\boxed{314\text{ original segmentation-training scenes}}
\]

are passed through the **frozen accepted 04H-CAL detector** and the same deterministic reliability / counterfactual / GeoEngine pipeline.

The existing accepted Notebook-07 attribution scenes are retained as held-out multimodal validation:

\[
\boxed{314\text{ train scenes} \quad+\quad 34\text{ validation scenes}}
\]

The final frozen geographic segmentation test remains:

\[
\boxed{58\text{ Pakistan + Spain scenes — LOCKED}}
\]

## Final corpus target

Every scene receives the same 10 deterministic instruction families:

1. predicted extent
2. uncertainty
3. overall sensor attribution
4. temporary-flood sensor attribution
5. spatial geometry
6. integrated evidence summary
7. review / abstention policy
8. physical-cause refusal
9. human-impact refusal
10. future-forecast refusal

Therefore the target is:

\[
314\times 10=3140\text{ training examples}
\]

and:

\[
34\times10=340\text{ held-out validation examples}
\]

for:

\[
\boxed{3480\text{ total grounded examples}}
\]

## Scientific constraints

- 04H-CAL only.
- 05B is never loaded.
- No segmentation training.
- Exact four-way TTA.
- Exact Notebook-05 confidence calibration.
- Exact Notebook-06 counterfactual semantics.
- No target/ground-truth field enters the language-model packets.
- Notebook-07 validation packets are reused unchanged.
- No external LLM creates the target answers.
- The geographic test is never constructed as a Dataset/DataLoader.

The later variational VLM therefore learns from **verified detector evidence**, not from fabricated captions or hidden ground-truth masks.


In [ ]:
# ============================================================
# CELL: P17-C00 — KAGGLE-COMPAT TERRATORCH INSTALL
# ============================================================
# Proven compatible path from 04E.
# IMPORTANT: on a fresh Kaggle image, pip may change NumPy to 1.26.4.
# If the next cell reports a NumPy/Pandas ABI error, restart the Python
# kernel ONCE, do NOT rerun this cell, then continue from P10-C01.

import sys, subprocess

print("Installing TerraTorch 1.1.1 (04E/04F-proven compatibility path)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "--no-cache-dir", "terratorch==1.1.1"
])
print("P17-C00 COMPLETE ✅")
print("If NumPy changed, restart the Python kernel once before P10-C01.")

In [ ]:
# ============================================================
# CELL: P17-C01 — ENVIRONMENT
# ============================================================
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys, gc, json, time, math, random, hashlib, zipfile, warnings, shutil, re
from pathlib import Path
from collections import Counter

import numpy as np
import scipy
from scipy import ndimage
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window
from rasterio.warp import transform as rio_transform

try:
    from pyproj import CRS
    PYPROJ_AVAILABLE = True
except Exception as e:
    PYPROJ_AVAILABLE = False
    warnings.warn(f"pyproj unavailable; metric area may be omitted: {e}")

from PIL import Image
import matplotlib.pyplot as plt

import terratorch
from terratorch.registry import BACKBONE_REGISTRY

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_08B_Full_Grounded_Corpus")

TRAIN_PACKET_ROOT = OUTPUT_ROOT / "train_packets"
DATASET_ROOT = OUTPUT_ROOT / "dataset"
IMAGE_ROOT = DATASET_ROOT / "images"
TRAIN_PANEL_ROOT = IMAGE_ROOT
METRIC_ROOT = OUTPUT_ROOT / "metrics"
EXTRACT_ROOT = OUTPUT_ROOT / "_extracted_inputs"

for p in [
    OUTPUT_ROOT,
    TRAIN_PACKET_ROOT,
    DATASET_ROOT,
    IMAGE_ROOT,
    METRIC_ROOT,
    EXTRACT_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

EXPECTED_07_ARCHIVE_SHA256 = (
    "1a8a5c35f2afedeb92aa58d7c161bca22bbaee762d6533709a515f3798a8344b"
)

print("=" * 112)
print("P17-C01 — FULL GROUNDED CORPUS ENVIRONMENT")
print("=" * 112)
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("SciPy       :", scipy.__version__)
print("PyTorch     :", torch.__version__)
print("Rasterio    :", rasterio.__version__)
print("Device      :", DEVICE)
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("Output root :", OUTPUT_ROOT)


In [ ]:
# ============================================================
# CELL: P17-C02 — LOCATE + VERIFY 04H-CAL / 05 / 06 / 07
# ============================================================
_file_cache = {}

def locate(filename, required=True):
    key = str(filename)
    if key in _file_cache:
        return _file_cache[key]

    matches = list(INPUT_ROOT.rglob(Path(filename).name))

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not locate '{filename}' under /kaggle/input."
            )
        return None

    selected = sorted(
        matches,
        key=lambda p: (len(str(p)), str(p)),
    )[0]

    _file_cache[key] = selected
    return selected

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def resolve_from_archive(filename, archive_name, required=True):
    direct = locate(filename, required=False)
    if direct is not None:
        return direct

    archive = locate(archive_name, required=False)

    if archive is None:
        if required:
            raise FileNotFoundError(
                f"Need either '{filename}' or '{archive_name}' in /kaggle/input."
            )
        return None

    out_dir = EXTRACT_ROOT / Path(archive_name).stem
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(out_dir)

    matches = list(out_dir.rglob(filename))

    if not matches:
        if required:
            raise FileNotFoundError(
                f"'{filename}' not found inside '{archive_name}'."
            )
        return None

    return matches[0]

# Frozen preprocessing.
FINAL_MANIFEST_PATH = locate("floodshift_preprocessing_manifest_v1.csv")
GLOBAL_LOCK_PATH = locate("floodshift_preprocessing_GLOBAL_LOCK_v1.json")
OPTICAL_NORM_PATH = locate("optical_10band_train_normalization_exact.csv")
SAR_SCENE_STATS_PATH = locate("train_raw_sar_scene_statistics.csv")

# Accepted detector.
CHECKPOINT_04H = locate("FloodShift_04H_best_mIoU.pt")
CAL_CONFIG_PATH = locate("FloodShift_04H_CAL_config.pt")

# Accepted stage-05 and stage-06 configs.
UNCERTAINTY_05_PATH = resolve_from_archive(
    "FloodShift_05_uncertainty_config.pt",
    "FloodShift_05_ARCHIVE.zip",
    required=True,
)

ATTR_CONFIG_06_PATH = resolve_from_archive(
    "FloodShift_06_attribution_config.pt",
    "FloodShift_06_ARCHIVE.zip",
    required=True,
)

EXPECTED_04H_SHA256 = "ef3341a7579f1316c80290dcd2a8f972c0adbf7e03e7a8c6bc239f30c4b47e53"
EXPECTED_04E_SHA256 = "6ab8ad8b41aa64b4ceee403539d8554a14f61cab235a0f0841ef13f77e390822"
EXPECTED_TYPE_BIAS = -0.15

SHA_04H = sha256_file(CHECKPOINT_04H)
assert SHA_04H == EXPECTED_04H_SHA256

cal_cfg = torch.load(
    CAL_CONFIG_PATH,
    map_location="cpu",
    weights_only=False,
)

unc05 = torch.load(
    UNCERTAINTY_05_PATH,
    map_location="cpu",
    weights_only=False,
)

attr06 = torch.load(
    ATTR_CONFIG_06_PATH,
    map_location="cpu",
    weights_only=False,
)

assert cal_cfg["source_04h_checkpoint_sha256"] == EXPECTED_04H_SHA256
assert cal_cfg["source_04e_sha256"] == EXPECTED_04E_SHA256
assert abs(float(cal_cfg["type_bias"]) - EXPECTED_TYPE_BIAS) < 1e-12
assert cal_cfg["decision"] == "04H_CAL_SINGLE_MODEL_PREFERRED"

assert unc05["source_04h_sha256"] == EXPECTED_04H_SHA256
assert abs(float(unc05["type_bias"]) - EXPECTED_TYPE_BIAS) < 1e-12

assert attr06["source_04h_sha256"] == EXPECTED_04H_SHA256
assert abs(float(attr06["type_bias"]) - EXPECTED_TYPE_BIAS) < 1e-12
assert abs(float(attr06["temperature"]) - float(unc05["temperature"])) < 1e-9

TYPE_BIAS = float(cal_cfg["type_bias"])
TEMPERATURE = float(unc05["temperature"])

ENTROPY_THRESHOLD_REVIEW_10 = float(
    unc05["entropy_threshold_review_10"]
)

ENTROPY_THRESHOLD_REVIEW_20 = float(
    unc05["entropy_threshold_review_20"]
)

HIGH_FRAGILITY_THRESHOLD = float(
    attr06["high_fragility_threshold"]
)

DOMINANCE_SHARE = float(
    attr06["dominance_share_threshold"]
)

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)

assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

# ------------------------------------------------------------
# Locate Notebook-07 validation source.
# Supports real ZIP or Kaggle auto-extracted dataset.
# ------------------------------------------------------------
ARCHIVE_07 = locate("FloodShift_07_ARCHIVE.zip", required=False)

if ARCHIVE_07 is not None:
    ARCHIVE_07_SHA = sha256_file(ARCHIVE_07)

    assert ARCHIVE_07_SHA == EXPECTED_07_ARCHIVE_SHA256, (
        f"Notebook-07 archive SHA mismatch: {ARCHIVE_07_SHA}"
    )

    ROOT_07 = EXTRACT_ROOT / "FloodShift_07_ARCHIVE"
    ROOT_07.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(ARCHIVE_07, "r") as zf:
        zf.extractall(ROOT_07)

    SOURCE_07_MODE = "verified_zip"

else:
    protocol07_match = locate(
        "FloodShift_07_protocol_record.json",
        required=False,
    )

    assert protocol07_match is not None, (
        "Attach FloodShift_07_ARCHIVE.zip or its Kaggle auto-extracted contents."
    )

    ROOT_07 = protocol07_match.parent
    ARCHIVE_07_SHA = EXPECTED_07_ARCHIVE_SHA256
    SOURCE_07_MODE = "kaggle_auto_extracted"

PACKETS_07_PATH = locate(
    "FloodShift_07_scene_evidence_packets.jsonl"
)

PROTOCOL_07_PATH = locate(
    "FloodShift_07_protocol_record.json"
)

SCHEMA_07_PATH = locate(
    "FloodShift_07_fact_schema.json"
)

with open(PROTOCOL_07_PATH, "r") as f:
    protocol07 = json.load(f)

assert protocol07["source_segmenter"] == "FloodShift-04H-CAL"
assert protocol07["05b_status"] == "NOT_PROMOTED_NOT_USED"
assert protocol07["packet_count"] == 34
assert protocol07["packet_verifier_failures"] == 0
assert protocol07["ground_truth_in_packet"] is False
assert protocol07["test_status"] == "LOCKED_NOT_EVALUATED"

print("=" * 112)
print("P17-C02 — FULL LINEAGE VERIFIED")
print("=" * 112)
print("04H SHA256          :", SHA_04H)
print("Type bias           :", TYPE_BIAS)
print("Temperature         :", TEMPERATURE)
print("Review-10 entropy   :", ENTROPY_THRESHOLD_REVIEW_10)
print("Review-20 entropy   :", ENTROPY_THRESHOLD_REVIEW_20)
print("Fragility threshold :", HIGH_FRAGILITY_THRESHOLD)
print("Notebook-07 mode    :", SOURCE_07_MODE)
print("Notebook-07 SHA     :", ARCHIVE_07_SHA)
print("05B                 : NOT USED")
print("Frozen geo test     : LOCKED_NOT_EVALUATED")
print("P17-C02 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C03 — MANIFEST + FROZEN TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = (
    manifest[manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    manifest[manifest["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    manifest[manifest["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# Original S2 source order:
# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = (
        np.isfinite(mu_all)
        & np.isfinite(sd_all)
        & (n_all > 0)
    )

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std(
    "vv_mean_db",
    "vv_std_db",
)

VH_MEAN, VH_STD = pooled_mean_std(
    "vh_mean_db",
    "vh_std_db",
)

SAR_MEAN = np.array(
    [VV_MEAN, VH_MEAN],
    dtype=np.float32,
)

SAR_STD = np.array(
    [VV_STD, VH_STD],
    dtype=np.float32,
)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 112)
print("P17-C03 — DATA CONFIG")
print("=" * 112)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : frozen continuous P(clear)")
print("Primary SAR input  : raw VV/VH")

In [ ]:
# ============================================================
# CELL: P17-C04 — TARGET + DATASET
#
# IMPORTANT:
# This intentionally preserves the CURRENT project's target mapping.
# It does NOT silently change label semantics mid-experiment.
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3

CLASS_NAMES = [
    "non_flood",
    "permanent_water",
    "temporary_flood",
]

TILE_SIZE = 256
TILES_PER_SCENE = 4

TILE_OFFSETS = [
    (0, 0),
    (0, 256),
    (256, 0),
    (256, 256),
]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(
            row["optical_reliability_path"]
        ),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(
        flood.shape,
        IGNORE_INDEX,
        dtype=np.uint8,
    )

    valid = flood != -1

    target[valid] = 0

    # Current frozen project semantics:
    # any valid JRC permanent-water pixel -> class 1
    target[
        valid & (jrc == 1)
    ] = 1

    # flood-positive but not JRC permanent water -> class 2
    target[
        valid
        & (jrc != 1)
        & (flood == 1)
    ] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(
        np.float32,
        copy=False,
    )

    valid = np.isfinite(sar).all(axis=0)

    out = np.zeros_like(
        sar,
        dtype=np.float32,
    )

    for c in range(2):
        m = valid & np.isfinite(sar[c])

        out[c, m] = (
            sar[c, m] - SAR_MEAN[c]
        ) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(
        np.float32,
        copy=False,
    )

    valid = np.all(
        np.isfinite(optical_13)
        & (optical_13 != 0),
        axis=0,
    )

    reflectance = optical_13 / 10000.0

    optical_10 = reflectance[
        OPTICAL_SOURCE_INDICES
    ]

    out = (
        optical_10
        - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(
        np.float32,
        copy=False,
    )

    # Frozen Notebook-03 convention.
    out[:, ~valid] = 0.0

    return out


def augment_all(
    sar,
    optical,
    reliability,
    target,
):
    # D4-aligned augmentation only.
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(
            reliability,
            axis=1,
        )
        target = np.flip(
            target,
            axis=1,
        )

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(
            reliability,
            axis=0,
        )
        target = np.flip(
            target,
            axis=0,
        )

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(
            sar,
            k=k,
            axes=(1, 2),
        )

        optical = np.rot90(
            optical,
            k=k,
            axes=(1, 2),
        )

        reliability = np.rot90(
            reliability,
            k=k,
            axes=(0, 1),
        )

        target = np.rot90(
            target,
            k=k,
            axes=(0, 1),
        )

    return (
        sar.copy(),
        optical.copy(),
        reliability.copy(),
        target.copy(),
    )


class FixedEvalDataset(Dataset):
    def __init__(
        self,
        dataframe,
        train_mode=False,
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.train_mode = train_mode

    def __len__(self):
        return (
            len(self.df)
            * TILES_PER_SCENE
        )

    def __getitem__(self, index):
        scene_index = (
            index // TILES_PER_SCENE
        )

        tile_index = (
            index % TILES_PER_SCENE
        )

        row = self.df.iloc[
            scene_index
        ]

        scene_id = str(
            row["scene_id"]
        )

        y0, x0 = TILE_OFFSETS[
            tile_index
        ]

        window = Window(
            x0,
            y0,
            TILE_SIZE,
            TILE_SIZE,
        )

        paths = resolve_scene_files(
            row
        )

        with rasterio.open(
            paths["sar"]
        ) as src:
            sar = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["optical"]
        ) as src:
            optical_13 = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["flood"]
        ) as src:
            flood = src.read(
                1,
                window=window,
            )

        with rasterio.open(
            paths["jrc"]
        ) as src:
            jrc = src.read(
                1,
                window=window,
            )

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r",
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0 + TILE_SIZE,
                x0:x0 + TILE_SIZE,
            ],
            dtype=np.float32,
        )

        sar = normalize_sar(sar)

        optical = normalize_optical(
            optical_13
        )

        target = build_three_class_target(
            flood,
            jrc,
        )

        reliability = np.clip(
            reliability,
            0.0,
            1.0,
        )

        if self.train_mode:
            (
                sar,
                optical,
                reliability,
                target,
            ) = augment_all(
                sar,
                optical,
                reliability,
                target,
            )

        return {
            "sar": torch.from_numpy(
                sar
            ),
            "optical": torch.from_numpy(
                optical
            ),
            "reliability": torch.from_numpy(
                reliability[
                    None,
                    ...,
                ]
            ),
            "target": torch.from_numpy(
                target.astype(
                    np.int64
                )
            ),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }

In [ ]:
# ============================================================
# CELL: P17-C05 — TRAIN-SCENE CORPUS INPUT
#
# Uses all 314 original segmentation-training scenes.
# No target/ground-truth values are consumed by the packet builder.
# ============================================================
NUM_WORKERS = 2
SCENE_BATCH_SIZE = 2

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

TRAIN_CORPUS_SCENES = set(
    train_df["scene_id"].astype(str).tolist()
)

# Load Notebook-07 held-out validation packets.
val_packets_07 = []

with open(PACKETS_07_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            val_packets_07.append(json.loads(line))

assert len(val_packets_07) == 34

VAL_CORPUS_SCENES = {
    p["scene_id"]
    for p in val_packets_07
}

assert TRAIN_CORPUS_SCENES.isdisjoint(VAL_CORPUS_SCENES)

TEST_SCENES = set(
    test_df["scene_id"].astype(str).tolist()
)

assert TRAIN_CORPUS_SCENES.isdisjoint(TEST_SCENES)
assert VAL_CORPUS_SCENES.isdisjoint(TEST_SCENES)

print("=" * 112)
print("P17-C05 — CORPUS SPLITS")
print("=" * 112)
print("Training scenes   :", len(TRAIN_CORPUS_SCENES))
print("Validation scenes :", len(VAL_CORPUS_SCENES))
print("Frozen test scenes:", len(TEST_SCENES))
print("Train/val overlap :", len(TRAIN_CORPUS_SCENES & VAL_CORPUS_SCENES))
print("Test constructed  : NO")
print("P17-C05 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C06 — FINAL TERRAMIND-RIFT ARCHITECTURE
# ============================================================

# ------------------------------------------------------------
# TerraMind pretrained input conventions
# Official Sen1Floods11 config statistics.
# ------------------------------------------------------------

TM_S2_ALL_MEAN = np.array(
    [
        2357.089,
        2137.385,
        2018.788,
        2082.986,
        2295.651,
        2854.537,
        3122.849,
        3040.560,
        3306.481,
        1473.847,
        506.070,
        2472.825,
        1838.929,
    ],
    dtype=np.float32,
)

TM_S2_ALL_STD = np.array(
    [
        1624.683,
        1675.806,
        1557.708,
        1833.702,
        1823.738,
        1733.977,
        1732.131,
        1679.732,
        1727.260,
        1024.687,
        442.165,
        1331.411,
        1160.419,
    ],
    dtype=np.float32,
)

TM_S2_SUBSET_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

TM_S2_MEAN = (
    TM_S2_ALL_MEAN[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S2_STD = (
    TM_S2_ALL_STD[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S1_MEAN = np.array(
    [-12.599, -20.293],
    dtype=np.float32,
)

TM_S1_STD = np.array(
    [5.195, 5.890],
    dtype=np.float32,
)

TM_OPTICAL_BAND_NAMES = [
    "BLUE",
    "GREEN",
    "RED",
    "RED_EDGE_1",
    "RED_EDGE_2",
    "RED_EDGE_3",
    "NIR_BROAD",
    "NIR_NARROW",
    "SWIR_1",
    "SWIR_2",
]

TM_LAYER_INDICES = [
    2,
    5,
    8,
    11,
]

TM_EMBED_DIM = 768
PYRAMID_CHANNELS = [
    64,
    128,
    256,
    512,
]

PYRAMID_SIZES = [
    128,
    64,
    32,
    16,
]


def make_group_norm(channels):
    for groups in [
        32,
        16,
        8,
        4,
        2,
        1,
    ]:
        if (
            channels
            % groups
            == 0
        ):
            return nn.GroupNorm(
                groups,
                channels,
            )

    raise RuntimeError(
        "No GroupNorm divisor."
    )


class ConvNormAct(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
    ):
        super().__init__()

        padding = (
            kernel_size // 2
        )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(
        self,
        channels,
        reduction=8,
    ):
        super().__init__()

        hidden = max(
            channels // reduction,
            8,
        )

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1,
        )

        self.fc2 = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1,
        )

    def forward(self, x):
        w = F.adaptive_avg_pool2d(
            x,
            1,
        )

        w = F.gelu(
            self.fc1(w)
        )

        w = torch.sigmoid(
            self.fc2(w)
        )

        return x * w


class ConvNeXtRefine(nn.Module):
    def __init__(
        self,
        channels,
        drop=0.0,
    ):
        super().__init__()

        self.dw = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )

        self.norm = make_group_norm(
            channels
        )

        self.pw1 = nn.Conv2d(
            channels,
            channels * 4,
            kernel_size=1,
        )

        self.pw2 = nn.Conv2d(
            channels * 4,
            channels,
            kernel_size=1,
        )

        self.drop = nn.Dropout2d(
            drop
        )

        self.se = SEBlock(
            channels
        )

    def forward(self, x):
        residual = x

        x = self.dw(x)

        x = self.norm(x)

        x = F.gelu(
            self.pw1(x)
        )

        x = self.drop(
            self.pw2(x)
        )

        x = self.se(x)

        return residual + x


class ModalityProjector(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        out_size,
    ):
        super().__init__()

        self.out_size = out_size

        self.proj = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
            ConvNeXtRefine(
                out_channels,
                drop=0.0,
            ),
        )

    def forward(self, x):
        x = self.proj(x)

        if (
            x.shape[-2:]
            != (
                self.out_size,
                self.out_size,
            )
        ):
            x = F.interpolate(
                x,
                size=(
                    self.out_size,
                    self.out_size,
                ),
                mode="bilinear",
                align_corners=False,
            )

        return x


class ReliabilityRouter(nn.Module):
    """
    Spatial SAR-optical router.

    P(clear) is used as a prior on the optical route,
    while the learned router can still override that prior.
    """

    def __init__(
        self,
        channels,
        hidden=64,
        prior_strength=0.50,
    ):
        super().__init__()

        hidden = min(
            hidden,
            max(
                channels // 2,
                16,
            ),
        )

        self.prior_strength = (
            float(
                prior_strength
            )
        )

        self.net = nn.Sequential(
            nn.Conv2d(
                channels * 2 + 1,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(
                hidden
            ),
            nn.GELU(),
            nn.Conv2d(
                hidden,
                2,
                kernel_size=1,
            ),
        )

        # Start near neutral learned routing.
        nn.init.zeros_(
            self.net[-1].weight
        )

        nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        r = F.interpolate(
            pclear,
            size=sar.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        r = r.clamp(
            0.0,
            1.0,
        )

        logits = self.net(
            torch.cat(
                [
                    sar,
                    optical,
                    r,
                ],
                dim=1,
            )
        )

        # Optical prior. Clamp avoids -inf at P(clear)=0.
        logits[
            :,
            1:2,
        ] = (
            logits[
                :,
                1:2,
            ]
            + self.prior_strength
            * torch.log(
                r.clamp(
                    min=0.05,
                    max=1.0,
                )
            )
        )

        weights = torch.softmax(
            logits,
            dim=1,
        )

        w_sar = weights[
            :,
            0:1,
        ]

        w_opt = weights[
            :,
            1:2,
        ]

        fused = (
            w_sar * sar
            + w_opt * optical
        )

        return (
            fused,
            weights,
        )


class DeepRIFT(nn.Module):
    """
    Zero-initialized reliability-conditioned cross attention.

    When P(clear) is high:
      SAR can borrow more optical context.

    When P(clear) is low:
      optical can borrow more SAR context.

    Output projections start at zero, so the block is an
    exact identity at initialization.
    """

    def __init__(
        self,
        channels=512,
        heads=8,
        dropout=0.0,
    ):
        super().__init__()

        self.channels = channels

        self.sar_norm = nn.LayerNorm(
            channels
        )

        self.opt_norm = nn.LayerNorm(
            channels
        )

        self.sar_from_opt = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.opt_from_sar = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.sar_out = nn.Linear(
            channels,
            channels,
        )

        self.opt_out = nn.Linear(
            channels,
            channels,
        )

        nn.init.zeros_(
            self.sar_out.weight
        )

        nn.init.zeros_(
            self.sar_out.bias
        )

        nn.init.zeros_(
            self.opt_out.weight
        )

        nn.init.zeros_(
            self.opt_out.bias
        )

    @staticmethod
    def _flatten(x):
        return (
            x.flatten(2)
            .transpose(1, 2)
            .contiguous()
        )

    @staticmethod
    def _unflatten(
        tokens,
        height,
        width,
    ):
        return (
            tokens
            .transpose(1, 2)
            .reshape(
                tokens.shape[0],
                tokens.shape[2],
                height,
                width,
            )
            .contiguous()
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        b, c, h, w = sar.shape

        sar_t = self._flatten(
            sar
        )

        opt_t = self._flatten(
            optical
        )

        r = F.adaptive_avg_pool2d(
            pclear,
            output_size=(h, w),
        )

        r_t = (
            r.flatten(2)
            .transpose(1, 2)
            .clamp(0.0, 1.0)
        )

        sar_n = self.sar_norm(
            sar_t
        )

        opt_n = self.opt_norm(
            opt_t
        )

        # Clear optical context receives larger K/V magnitude.
        opt_context = (
            opt_n
            * (
                0.25
                + 0.75 * r_t
            )
        )

        sar_delta, _ = (
            self.sar_from_opt(
                query=sar_n,
                key=opt_context,
                value=opt_context,
                need_weights=False,
            )
        )

        # Under low P(clear), the optical branch borrows SAR more.
        opt_delta, _ = (
            self.opt_from_sar(
                query=opt_n,
                key=sar_n,
                value=sar_n,
                need_weights=False,
            )
        )

        sar_t = (
            sar_t
            + r_t
            * self.sar_out(
                sar_delta
            )
        )

        opt_t = (
            opt_t
            + (1.0 - r_t)
            * self.opt_out(
                opt_delta
            )
        )

        sar = self._unflatten(
            sar_t,
            h,
            w,
        )

        optical = self._unflatten(
            opt_t,
            h,
            w,
        )

        return sar, optical


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.fuse = nn.Sequential(
            ConvNormAct(
                in_channels
                + skip_channels,
                out_channels,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        x = torch.cat(
            [
                x,
                skip,
            ],
            dim=1,
        )

        return self.fuse(x)


class FloodShiftTerraMindRIFT(nn.Module):
    def __init__(
        self,
        num_classes=3,
        pretrained_backbone=False,
    ):
        super().__init__()

        # concat is used deliberately:
        # each spatial token contains S2 and S1 embeddings
        # concatenated in the modality order supplied below.
        self.backbone = (
            BACKBONE_REGISTRY.build(
                "terramind_v1_base",
                pretrained=pretrained_backbone,
                img_size=TILE_SIZE,
                modalities=[
                    "S2L1C",
                    "S1GRD",
                ],
                bands={
                    "S2L1C":
                        TM_OPTICAL_BAND_NAMES,
                },
                merge_method="concat",
            )
        )

        self.register_buffer(
            "project_opt_mean",
            torch.tensor(
                OPTICAL_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_opt_std",
            torch.tensor(
                OPTICAL_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_mean",
            torch.tensor(
                SAR_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_std",
            torch.tensor(
                SAR_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_mean",
            torch.tensor(
                TM_S2_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_std",
            torch.tensor(
                TM_S2_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_mean",
            torch.tensor(
                TM_S1_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_std",
            torch.tensor(
                TM_S1_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.sar_projectors = (
            nn.ModuleList()
        )

        self.opt_projectors = (
            nn.ModuleList()
        )

        self.routers = nn.ModuleList()

        for (
            channels,
            out_size,
        ) in zip(
            PYRAMID_CHANNELS,
            PYRAMID_SIZES,
        ):
            self.sar_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.opt_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.routers.append(
                ReliabilityRouter(
                    channels,
                    prior_strength=0.50,
                )
            )

        self.deep_rift = DeepRIFT(
            channels=PYRAMID_CHANNELS[-1],
            heads=8,
            dropout=0.0,
        )

        self.deep_context = nn.Sequential(
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
        )

        self.decoder3 = DecoderBlock(
            512,
            256,
            256,
        )

        self.decoder2 = DecoderBlock(
            256,
            128,
            128,
        )

        self.decoder1 = DecoderBlock(
            128,
            64,
            64,
        )

        self.final_refine = nn.Sequential(
            ConvNormAct(
                64,
                64,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                64,
                drop=0.05,
            ),
        )

        self.segmentation_head = (
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            )
        )

        self.aux_head = nn.Sequential(
            nn.Conv2d(
                128,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(64),
            nn.GELU(),
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            ),
        )

    def to_terramind_inputs(
        self,
        sar_z,
        optical_z,
    ):
        # Detect frozen invalid-fill pixels BEFORE inverse affine.
        opt_invalid = (
            optical_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        sar_invalid = (
            sar_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        # Frozen optical z-score -> reflectance -> DN.
        optical_reflectance = (
            optical_z
            * self.project_opt_std
            + self.project_opt_mean
        )

        optical_dn = (
            optical_reflectance
            * 10000.0
        )

        optical_tm = (
            optical_dn
            - self.tm_opt_mean
        ) / self.tm_opt_std

        # Frozen SAR z-score -> raw dB.
        sar_db = (
            sar_z
            * self.project_sar_std
            + self.project_sar_mean
        )

        sar_tm = (
            sar_db
            - self.tm_sar_mean
        ) / self.tm_sar_std

        # Preserve invalid-fill convention in model space.
        optical_tm = optical_tm.masked_fill(
            opt_invalid,
            0.0,
        )

        sar_tm = sar_tm.masked_fill(
            sar_invalid,
            0.0,
        )

        return (
            sar_tm,
            optical_tm,
        )

    @staticmethod
    def tokens_to_map(tokens):
        b, n, c = tokens.shape

        side = int(
            math.isqrt(n)
        )

        if (
            side * side
            != n
        ):
            raise RuntimeError(
                f"TerraMind token count {n} "
                "is not a square grid."
            )

        return (
            tokens
            .transpose(1, 2)
            .reshape(
                b,
                c,
                side,
                side,
            )
            .contiguous()
        )

    def extract_modality_tokens(
        self,
        tm_outputs,
    ):
        if (
            not isinstance(
                tm_outputs,
                (list, tuple),
            )
            or len(tm_outputs) < 12
        ):
            raise RuntimeError(
                "Unexpected TerraMind output. "
                "Expected list of 12 transformer-layer outputs."
            )

        sar_tokens = []
        opt_tokens = []

        for layer_index in TM_LAYER_INDICES:
            tokens = tm_outputs[
                layer_index
            ]

            if (
                not torch.is_tensor(tokens)
                or tokens.ndim != 3
            ):
                raise RuntimeError(
                    "TerraMind concat output should be "
                    "[B, tokens, embedding]."
                )

            expected_dim = (
                TM_EMBED_DIM * 2
            )

            if (
                tokens.shape[-1]
                != expected_dim
            ):
                raise RuntimeError(
                    "Expected concat embedding dimension "
                    f"{expected_dim}, got "
                    f"{tokens.shape[-1]}. "
                    "Check TerraTorch version and merge_method."
                )

            # modalities were supplied in order:
            # [S2L1C, S1GRD]
            optical_part = tokens[
                :,
                :,
                :TM_EMBED_DIM,
            ]

            sar_part = tokens[
                :,
                :,
                TM_EMBED_DIM:,
            ]

            opt_tokens.append(
                optical_part
            )

            sar_tokens.append(
                sar_part
            )

        return (
            sar_tokens,
            opt_tokens,
        )

    def forward(
        self,
        sar,
        optical,
        reliability,
        return_aux=False,
    ):
        (
            sar_tm,
            optical_tm,
        ) = self.to_terramind_inputs(
            sar,
            optical,
        )

        tm_outputs = self.backbone(
            {
                "S2L1C": optical_tm,
                "S1GRD": sar_tm,
            }
        )

        (
            sar_tokens,
            opt_tokens,
        ) = self.extract_modality_tokens(
            tm_outputs
        )

        sar_maps = []
        opt_maps = []

        for i in range(4):
            sar_map = (
                self.tokens_to_map(
                    sar_tokens[i]
                )
            )

            opt_map = (
                self.tokens_to_map(
                    opt_tokens[i]
                )
            )

            sar_map = (
                self.sar_projectors[i](
                    sar_map
                )
            )

            opt_map = (
                self.opt_projectors[i](
                    opt_map
                )
            )

            sar_maps.append(
                sar_map
            )

            opt_maps.append(
                opt_map
            )

        # RIFT only at deepest 16x16 level.
        (
            sar_maps[-1],
            opt_maps[-1],
        ) = self.deep_rift(
            sar_maps[-1],
            opt_maps[-1],
            reliability,
        )

        fused = []
        router_weights = []

        for i in range(4):
            (
                fused_i,
                weights_i,
            ) = self.routers[i](
                sar_maps[i],
                opt_maps[i],
                reliability,
            )

            fused.append(
                fused_i
            )

            router_weights.append(
                weights_i
            )

        f0, f1, f2, f3 = fused

        x = self.deep_context(
            f3
        )

        x = self.decoder3(
            x,
            f2,
        )

        x = self.decoder2(
            x,
            f1,
        )

        aux_logits = self.aux_head(
            x
        )

        x = self.decoder1(
            x,
            f0,
        )

        x = F.interpolate(
            x,
            size=(
                TILE_SIZE,
                TILE_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        x = self.final_refine(
            x
        )

        logits = (
            self.segmentation_head(
                x
            )
        )

        if return_aux:
            return {
                "logits": logits,
                "aux_logits": aux_logits,
                "router_weights": router_weights,
            }

        return logits

In [ ]:
# ============================================================
# CELL: P17-C07 — REBUILD EXACT ACCEPTED 04H-CAL MODEL
# ============================================================
class HierarchicalResidualHead(nn.Module):
    def __init__(self, in_channels=16, hidden=48, max_delta=1.50):
        super().__init__()
        self.max_delta = float(max_delta)
        self.net = nn.Sequential(
            ConvNormAct(in_channels, hidden, 3),
            ConvNeXtRefine(hidden, drop=0.05),
            ConvNormAct(hidden, hidden, 3),
            nn.Conv2d(hidden, 2, kernel_size=1),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, base_prob, sar, optical, reliability):
        x = torch.cat([base_prob, sar, optical, reliability], dim=1)
        raw = self.net(x).float()
        return self.max_delta * torch.tanh(raw / self.max_delta)

class FloodShift04H(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.head = HierarchicalResidualHead()

        for p in self.base.parameters():
            p.requires_grad = False
        self.base.eval()

    @property
    def backbone(self):
        return self.base.backbone

    def train(self, mode=True):
        super().train(mode)
        self.base.eval()
        self.head.train(mode)
        return self

    def forward(self, sar, optical, reliability, return_aux=False):
        with torch.no_grad():
            base_out = self.base(
                sar,
                optical,
                reliability,
                return_aux=True,
            )
            base_logits = base_out["logits"].float()
            base_prob = torch.softmax(base_logits, dim=1)

        correction = self.head(
            base_prob.to(sar.dtype),
            sar,
            optical,
            reliability,
        )

        delta_water = correction[:, 0]
        delta_type = correction[:, 1]

        base_water_logit = (
            torch.logsumexp(base_logits[:, 1:3], dim=1)
            - base_logits[:, 0]
        )
        base_type_logit = base_logits[:, 2] - base_logits[:, 1]

        water_logit = base_water_logit + delta_water
        type_logit = base_type_logit + delta_type

        p_water = torch.sigmoid(water_logit)
        p_temp_given_water = torch.sigmoid(type_logit)

        p0 = 1.0 - p_water
        p1 = p_water * (1.0 - p_temp_given_water)
        p2 = p_water * p_temp_given_water

        probs = torch.stack([p0, p1, p2], dim=1).clamp_min(1e-7)
        log_probs = probs.log()

        if return_aux:
            return {
                "logits": log_probs,
                "probs": probs,
                "base_logits": base_logits,
                "base_prob": base_prob,
                "base_water_logit": base_water_logit,
                "base_type_logit": base_type_logit,
                "water_logit": water_logit,
                "type_logit": type_logit,
                "delta_water": delta_water,
                "delta_type": delta_type,
            }

        return log_probs

payload = torch.load(CHECKPOINT_04H, map_location="cpu", weights_only=False)

assert payload.get("base_sha256") == EXPECTED_04E_SHA256
assert int(payload.get("epoch", -1)) == 4
assert payload.get("tag") == "best_mIoU"

base = FloodShiftTerraMindRIFT(
    num_classes=NUM_CLASSES,
    pretrained_backbone=False,
)
model = FloodShift04H(base)

missing, unexpected = model.load_state_dict(
    payload["state_dict"],
    strict=False,
)

assert not missing, f"Missing keys: {missing[:10]}"
assert not unexpected, f"Unexpected keys: {unexpected[:10]}"

bad = [
    n for n, t in model.state_dict().items()
    if torch.is_floating_point(t) and not torch.isfinite(t).all()
]
assert not bad

model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad = False

print("=" * 112)
print("P17-C07 — ACCEPTED 04H-CAL MODEL READY")
print("=" * 112)
print("Epoch          :", payload["epoch"])
print("Tag            :", payload["tag"])
print("Type bias      :", TYPE_BIAS)
print("Temperature    :", TEMPERATURE)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("P17-C07 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C08 — METRICS + FROZEN 4-WAY TTA + COUNTERFACTUALS
# ============================================================
TTA_MODES = ["identity", "hflip", "vflip", "rot180"]

def confusion_from_predictions(pred, target):
    valid = target != IGNORE_INDEX
    target_v = target[valid].long()
    pred_v = pred[valid].long()

    indices = NUM_CLASSES * target_v + pred_v
    return torch.bincount(
        indices,
        minlength=NUM_CLASSES * NUM_CLASSES,
    ).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics_from_confusion(cm):
    if torch.is_tensor(cm):
        cm = cm.detach().cpu().numpy().astype(np.float64)
    else:
        cm = np.asarray(cm, dtype=np.float64)

    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    den_iou = tp + fp + fn
    iou = np.divide(
        tp,
        den_iou,
        out=np.full_like(tp, np.nan),
        where=den_iou > 0,
    )

    accuracy = float(tp.sum() / max(cm.sum(), 1.0))

    out = {
        "mIoU": float(np.nanmean(iou)),
        "accuracy": accuracy,
        "confusion": cm.tolist(),
    }

    for i, name in enumerate(CLASS_NAMES):
        out[f"{name}_IoU"] = float(iou[i])

    return out

def compact_metrics(m):
    return {
        "mIoU": round(m["mIoU"], 4),
        "non_flood": round(m["non_flood_IoU"], 4),
        "permanent": round(m["permanent_water_IoU"], 4),
        "temporary": round(m["temporary_flood_IoU"], 4),
    }

def _tta_transform(x, mode):
    if mode == "identity":
        return x
    if mode == "hflip":
        return torch.flip(x, dims=[-1])
    if mode == "vflip":
        return torch.flip(x, dims=[-2])
    if mode == "rot180":
        return torch.rot90(x, k=2, dims=(-2, -1))
    raise ValueError(mode)

def _tta_inverse(x, mode):
    return _tta_transform(x, mode)

def calibrated_log_probs_from_aux(aux):
    water_logit = aux["water_logit"].float()
    type_logit = aux["type_logit"].float() + TYPE_BIAS

    p_water = torch.sigmoid(water_logit)
    p_temp_given_water = torch.sigmoid(type_logit)

    p0 = 1.0 - p_water
    p1 = p_water * (1.0 - p_temp_given_water)
    p2 = p_water * p_temp_given_water

    probs = torch.stack([p0, p1, p2], dim=1).clamp_min(1e-7)
    return probs.log()

@torch.inference_mode()
def tta_log_probs_variant(model_to_eval, sar, optical, reliability, variant):
    if variant == "full":
        sar_v = sar
        opt_v = optical
        rel_v = reliability

    elif variant == "no_sar":
        sar_v = torch.zeros_like(sar)
        opt_v = optical
        rel_v = reliability

    elif variant == "no_optical":
        sar_v = sar
        opt_v = torch.zeros_like(optical)
        rel_v = torch.zeros_like(reliability)

    else:
        raise ValueError(variant)

    total = None

    for mode in TTA_MODES:
        sar_t = _tta_transform(sar_v, mode)
        opt_t = _tta_transform(opt_v, mode)
        rel_t = _tta_transform(rel_v, mode)

        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            aux = model_to_eval(
                sar_t,
                opt_t,
                rel_t,
                return_aux=True,
            )

        lp = calibrated_log_probs_from_aux(aux)
        lp = _tta_inverse(lp, mode).float()

        total = lp if total is None else total + lp

    return total / len(TTA_MODES)

def probs_with_temperature(log_probs):
    return torch.softmax(
        log_probs.float() / TEMPERATURE,
        dim=1,
    )

def normalized_entropy(probs):
    return (
        -(probs * probs.clamp_min(1e-8).log()).sum(dim=1)
        / math.log(NUM_CLASSES)
    )

def normalized_js(p, q):
    """
    Jensen-Shannon divergence normalized by ln(2), so output is in [0,1].
    Shapes: [B,C,H,W] -> [B,H,W]
    """
    p = p.clamp_min(1e-8)
    q = q.clamp_min(1e-8)
    m = 0.5 * (p + q)

    kl_pm = (p * (p.log() - m.log())).sum(dim=1)
    kl_qm = (q * (q.log() - m.log())).sum(dim=1)

    js = 0.5 * (kl_pm + kl_qm)
    return (js / math.log(2.0)).clamp(0.0, 1.0)

print("P17-C08 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C09 — DETERMINISTIC GEOSPATIAL HELPERS
# ============================================================
LOW_PCLEAR_THRESHOLD = 0.50
PACKET_SCHEMA_VERSION = "FloodShift-GeoEvidence-v1"

def finite_or_none(x):
    if x is None:
        return None
    x = float(x)
    return x if np.isfinite(x) else None

def quantile_or_none(x, q):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return None
    return float(np.quantile(x, q))

def pixel_area_map_km2(transform, crs, height, width):
    """
    Deterministic area map.

    Projected CRS:
      affine cell area × CRS linear-unit conversion.

    Geographic CRS:
      row-wise geodesic cell area when grid has no rotation.

    Returns:
      area_map_km2 or None,
      method string.
    """
    if crs is None or not PYPROJ_AVAILABLE:
        return None, "unavailable"

    crs_obj = CRS.from_user_input(crs)

    if crs_obj.is_projected:
        det = abs(
            transform.a * transform.e
            - transform.b * transform.d
        )

        axis = crs_obj.axis_info
        fx = (
            float(axis[0].unit_conversion_factor)
            if len(axis) >= 1 and axis[0].unit_conversion_factor
            else 1.0
        )
        fy = (
            float(axis[1].unit_conversion_factor)
            if len(axis) >= 2 and axis[1].unit_conversion_factor
            else fx
        )

        area_km2 = det * fx * fy / 1_000_000.0

        return (
            np.full(
                (height, width),
                area_km2,
                dtype=np.float64,
            ),
            "projected_affine_exact",
        )

    if crs_obj.is_geographic:
        # Normal north-up geographic raster.
        if abs(transform.b) > 1e-12 or abs(transform.d) > 1e-12:
            return None, "geographic_rotated_grid_unsupported"

        geod = crs_obj.get_geod()
        out = np.zeros((height, width), dtype=np.float64)

        x0 = float(transform.c)
        x1 = float(transform.c + transform.a)

        for row in range(height):
            y0 = float(transform.f + row * transform.e)
            y1 = float(transform.f + (row + 1) * transform.e)

            area_m2, _ = geod.polygon_area_perimeter(
                [x0, x1, x1, x0],
                [y0, y0, y1, y1],
            )

            out[row, :] = abs(area_m2) / 1_000_000.0

        return out, "geographic_geodesic_row_exact"

    return None, "unsupported_crs"

def centroid_wgs84(mask, transform, crs):
    rows, cols = np.nonzero(mask)
    if len(rows) == 0:
        return None

    r = float(rows.mean())
    c = float(cols.mean())

    x, y = rasterio.transform.xy(
        transform,
        r,
        c,
        offset="center",
    )

    if crs is None:
        return {
            "row": r,
            "col": c,
            "longitude": None,
            "latitude": None,
        }

    try:
        lon, lat = rio_transform(
            crs,
            "EPSG:4326",
            [x],
            [y],
        )

        return {
            "row": r,
            "col": c,
            "longitude": float(lon[0]),
            "latitude": float(lat[0]),
        }

    except Exception:
        return {
            "row": r,
            "col": c,
            "longitude": None,
            "latitude": None,
        }

def component_geometry(temp_mask, area_map, transform, crs):
    structure = np.ones((3, 3), dtype=np.uint8)
    labels, n_components = ndimage.label(
        temp_mask.astype(bool),
        structure=structure,
    )

    if n_components == 0:
        return {
            "component_count": 0,
            "largest_component_pixels": 0,
            "largest_component_area_km2": 0.0 if area_map is not None else None,
            "largest_component_fraction_of_temp": 0.0,
            "largest_component_centroid": None,
        }

    total_temp = int(temp_mask.sum())

    best_label = None
    best_pixels = -1
    best_area = -1.0

    for lab in range(1, n_components + 1):
        m = labels == lab
        pixels = int(m.sum())

        if area_map is not None:
            area = float(area_map[m].sum())
            score = area
        else:
            area = None
            score = float(pixels)

        if score > best_area:
            best_area = score
            best_label = lab
            best_pixels = pixels
            best_area_km2 = area

    best_mask = labels == best_label

    return {
        "component_count": int(n_components),
        "largest_component_pixels": int(best_pixels),
        "largest_component_area_km2": finite_or_none(best_area_km2),
        "largest_component_fraction_of_temp": (
            float(best_pixels / total_temp)
            if total_temp > 0
            else 0.0
        ),
        "largest_component_centroid": centroid_wgs84(
            best_mask,
            transform,
            crs,
        ),
    }

def dominance_label(sar_share, optical_share):
    if sar_share >= DOMINANCE_SHARE:
        return "SAR-dominant"
    if optical_share >= DOMINANCE_SHARE:
        return "Optical-dominant"
    return "Balanced"

print("P17-C09 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C10 — STREAMING SCENE INFERENCE HELPERS
#
# One scene is processed at a time to avoid retaining 314 large
# multimodal map sets in RAM.
# ============================================================
SCENE_H = 512
SCENE_W = 512
LOW_PCLEAR_THRESHOLD = 0.50
PACKET_SCHEMA_VERSION = "FloodShift-GeoEvidence-v1"

def init_scene_maps():
    return {
        "prediction": np.zeros((SCENE_H, SCENE_W), dtype=np.uint8),
        "confidence": np.zeros((SCENE_H, SCENE_W), dtype=np.float32),
        "entropy": np.zeros((SCENE_H, SCENE_W), dtype=np.float32),
        "pclear": np.zeros((SCENE_H, SCENE_W), dtype=np.float32),
        "js_sar": np.zeros((SCENE_H, SCENE_W), dtype=np.float32),
        "js_optical": np.zeros((SCENE_H, SCENE_W), dtype=np.float32),
        "flip_no_sar": np.zeros((SCENE_H, SCENE_W), dtype=np.uint8),
        "flip_no_optical": np.zeros((SCENE_H, SCENE_W), dtype=np.uint8),
        "coverage": np.zeros((SCENE_H, SCENE_W), dtype=np.uint8),
        "sar_vv_db": np.zeros((SCENE_H, SCENE_W), dtype=np.float32),
        "rgb_reflectance": np.zeros((SCENE_H, SCENE_W, 3), dtype=np.float32),
    }

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

@torch.inference_mode()
def infer_one_scene(scene_row):
    one_df = pd.DataFrame([scene_row])

    ds = FixedEvalDataset(
        one_df,
        train_mode=False,
    )

    loader = DataLoader(
        ds,
        batch_size=SCENE_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        persistent_workers=False,
    )

    sm = init_scene_maps()

    for batch in loader:
        sar = batch["sar"].to(
            DEVICE,
            non_blocking=True,
        )

        opt = batch["optical"].to(
            DEVICE,
            non_blocking=True,
        )

        rel = batch["reliability"].to(
            DEVICE,
            non_blocking=True,
        )

        lp_full = tta_log_probs_variant(
            model,
            sar,
            opt,
            rel,
            "full",
        )

        lp_no_sar = tta_log_probs_variant(
            model,
            sar,
            opt,
            rel,
            "no_sar",
        )

        lp_no_opt = tta_log_probs_variant(
            model,
            sar,
            opt,
            rel,
            "no_optical",
        )

        p_full = probs_with_temperature(lp_full)
        p_no_sar = probs_with_temperature(lp_no_sar)
        p_no_opt = probs_with_temperature(lp_no_opt)

        pred_full = p_full.argmax(1)
        pred_no_sar = p_no_sar.argmax(1)
        pred_no_opt = p_no_opt.argmax(1)

        entropy = normalized_entropy(p_full)
        confidence = p_full.max(dim=1).values

        js_sar = normalized_js(
            p_full,
            p_no_sar,
        )

        js_opt = normalized_js(
            p_full,
            p_no_opt,
        )

        sar_present = (
            sar.abs().sum(dim=1) > 1e-12
        )

        opt_present = (
            opt.abs().sum(dim=1) > 1e-12
        )

        coverage = (
            sar_present
            | opt_present
        )

        tile_indices = batch["tile_index"]

        for i in range(sar.shape[0]):
            tid = int(tile_indices[i])

            y0, x0 = TILE_OFFSETS[tid]

            ys = slice(
                y0,
                y0 + TILE_SIZE,
            )

            xs = slice(
                x0,
                x0 + TILE_SIZE,
            )

            pred_np = (
                pred_full[i]
                .detach()
                .cpu()
                .numpy()
                .astype(np.uint8)
            )

            sar_np = (
                sar[i]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            opt_np = (
                opt[i]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            vv_db = (
                sar_np[0]
                * float(SAR_STD[0])
                + float(SAR_MEAN[0])
            )

            optical_ref = (
                opt_np
                * OPTICAL_STD[:, None, None]
                + OPTICAL_MEAN[:, None, None]
            )

            rgb = np.stack(
                [
                    optical_ref[2],
                    optical_ref[1],
                    optical_ref[0],
                ],
                axis=-1,
            ).astype(np.float32)

            sm["prediction"][ys, xs] = pred_np

            sm["confidence"][ys, xs] = (
                confidence[i]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            sm["entropy"][ys, xs] = (
                entropy[i]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            sm["pclear"][ys, xs] = (
                rel[i, 0]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            sm["js_sar"][ys, xs] = (
                js_sar[i]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            sm["js_optical"][ys, xs] = (
                js_opt[i]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            sm["flip_no_sar"][ys, xs] = (
                pred_no_sar[i] != pred_full[i]
            ).detach().cpu().numpy().astype(np.uint8)

            sm["flip_no_optical"][ys, xs] = (
                pred_no_opt[i] != pred_full[i]
            ).detach().cpu().numpy().astype(np.uint8)

            sm["coverage"][ys, xs] = (
                coverage[i]
                .detach()
                .cpu()
                .numpy()
                .astype(np.uint8)
            )

            sm["sar_vv_db"][ys, xs] = vv_db
            sm["rgb_reflectance"][ys, xs] = rgb

    return sm

print("P17-C10 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C11 — PACKET + PANEL BUILDERS
# ============================================================
def stretch_channel(x, mask):
    vals = x[mask & np.isfinite(x)]

    if vals.size == 0:
        return np.zeros_like(
            x,
            dtype=np.float32,
        )

    lo = np.quantile(vals, 0.02)
    hi = np.quantile(vals, 0.98)

    if hi <= lo:
        return np.zeros_like(
            x,
            dtype=np.float32,
        )

    return np.clip(
        (x - lo) / (hi - lo),
        0.0,
        1.0,
    ).astype(np.float32)

def stretch_rgb(rgb, mask):
    out = np.zeros_like(
        rgb,
        dtype=np.float32,
    )

    for c in range(3):
        out[..., c] = stretch_channel(
            rgb[..., c],
            mask,
        )

    out[~mask] = 0.0
    return out

def build_packet(scene_id, scene_row, sm):
    paths = resolve_scene_files(
        scene_row
    )

    with rasterio.open(
        paths["sar"]
    ) as src:

        scene_transform = src.transform
        scene_crs = src.crs

    area_map, area_method = pixel_area_map_km2(
        scene_transform,
        scene_crs,
        SCENE_H,
        SCENE_W,
    )

    coverage = (
        sm["coverage"]
        .astype(bool)
    )

    pred = sm["prediction"]

    n_coverage = int(
        coverage.sum()
    )

    assert n_coverage > 0

    nonflood = (
        coverage
        & (pred == 0)
    )

    permanent = (
        coverage
        & (pred == 1)
    )

    temporary = (
        coverage
        & (pred == 2)
    )

    water = (
        permanent
        | temporary
    )

    entropy = sm["entropy"]
    pclear = sm["pclear"]
    js_sar = sm["js_sar"]
    js_opt = sm["js_optical"]

    fragility = np.maximum(
        js_sar,
        js_opt,
    )

    mean_js_sar = float(
        js_sar[coverage].mean()
    )

    mean_js_opt = float(
        js_opt[coverage].mean()
    )

    denom = (
        mean_js_sar
        + mean_js_opt
    )

    if denom > 1e-12:
        sar_share = (
            mean_js_sar
            / denom
        )

        optical_share = (
            mean_js_opt
            / denom
        )
    else:
        sar_share = 0.5
        optical_share = 0.5

    tile_fragility_means = []

    for tid, (y0, x0) in enumerate(TILE_OFFSETS):
        ys = slice(
            y0,
            y0 + TILE_SIZE,
        )

        xs = slice(
            x0,
            x0 + TILE_SIZE,
        )

        cov_t = coverage[ys, xs]

        if cov_t.any():
            tile_fragility_means.append(
                float(
                    fragility[ys, xs][cov_t].mean()
                )
            )

    high_fragility_tile_fraction = float(
        np.mean(
            np.asarray(
                tile_fragility_means
            )
            >= HIGH_FRAGILITY_THRESHOLD
        )
    )

    if area_map is not None:
        coverage_area = float(
            area_map[coverage].sum()
        )

        nonflood_area = float(
            area_map[nonflood].sum()
        )

        permanent_area = float(
            area_map[permanent].sum()
        )

        temporary_area = float(
            area_map[temporary].sum()
        )

        water_area = float(
            area_map[water].sum()
        )
    else:
        coverage_area = None
        nonflood_area = None
        permanent_area = None
        temporary_area = None
        water_area = None

    temp_geom = component_geometry(
        temporary,
        area_map,
        scene_transform,
        scene_crs,
    )

    def masked_mean(arr, mask):
        if not mask.any():
            return None

        return float(
            arr[mask].mean()
        )

    packet = {
        "schema_version": PACKET_SCHEMA_VERSION,
        "scene_id": scene_id,

        "provenance": {
            "segmenter": "FloodShift-04H-CAL",
            "source_04h_sha256": SHA_04H,
            "source_04e_sha256": EXPECTED_04E_SHA256,
            "type_bias": TYPE_BIAS,
            "temperature": TEMPERATURE,
            "uncertainty_source": "FloodShift-05",
            "attribution_source": "FloodShift-06",
            "corpus_role": "vlm_training",
            "test_status": "LOCKED_NOT_EVALUATED",
        },

        "geospatial_reference": {
            "crs": (
                str(scene_crs)
                if scene_crs is not None
                else None
            ),
            "raster_width": int(SCENE_W),
            "raster_height": int(SCENE_H),
            "area_method": area_method,
        },

        "coverage": {
            "covered_pixels": n_coverage,
            "total_grid_pixels": int(
                SCENE_H * SCENE_W
            ),
            "coverage_fraction": float(
                n_coverage
                / (SCENE_H * SCENE_W)
            ),
            "coverage_area_km2": finite_or_none(
                coverage_area
            ),
        },

        "segmentation": {
            "non_flood": {
                "pixels": int(
                    nonflood.sum()
                ),
                "fraction_of_coverage": float(
                    nonflood.sum()
                    / n_coverage
                ),
                "area_km2": finite_or_none(
                    nonflood_area
                ),
            },

            "permanent_water": {
                "pixels": int(
                    permanent.sum()
                ),
                "fraction_of_coverage": float(
                    permanent.sum()
                    / n_coverage
                ),
                "area_km2": finite_or_none(
                    permanent_area
                ),
            },

            "temporary_flood": {
                "pixels": int(
                    temporary.sum()
                ),
                "fraction_of_coverage": float(
                    temporary.sum()
                    / n_coverage
                ),
                "area_km2": finite_or_none(
                    temporary_area
                ),
            },

            "predicted_water_total": {
                "pixels": int(
                    water.sum()
                ),
                "fraction_of_coverage": float(
                    water.sum()
                    / n_coverage
                ),
                "area_km2": finite_or_none(
                    water_area
                ),
            },
        },

        "uncertainty": {
            "score": "normalized_predictive_entropy",
            "mean_entropy": float(
                entropy[coverage].mean()
            ),
            "median_entropy": float(
                np.median(
                    entropy[coverage]
                )
            ),
            "p90_entropy": quantile_or_none(
                entropy[coverage],
                0.90,
            ),
            "review10_threshold": ENTROPY_THRESHOLD_REVIEW_10,
            "review20_threshold": ENTROPY_THRESHOLD_REVIEW_20,
            "review10_fraction": float(
                np.mean(
                    entropy[coverage]
                    > ENTROPY_THRESHOLD_REVIEW_10
                )
            ),
            "review20_fraction": float(
                np.mean(
                    entropy[coverage]
                    > ENTROPY_THRESHOLD_REVIEW_20
                )
            ),
            "temporary_flood_mean_entropy": masked_mean(
                entropy,
                temporary,
            ),
        },

        "sensor_quality": {
            "optical_reliability_definition": "P(clear)",
            "mean_pclear": float(
                pclear[coverage].mean()
            ),
            "median_pclear": float(
                np.median(
                    pclear[coverage]
                )
            ),
            "low_pclear_threshold": LOW_PCLEAR_THRESHOLD,
            "low_pclear_fraction": float(
                np.mean(
                    pclear[coverage]
                    < LOW_PCLEAR_THRESHOLD
                )
            ),
        },

        "counterfactual_attribution": {
            "metric": "normalized_Jensen-Shannon_divergence",
            "interpretation": (
                "model dependence / counterfactual sensitivity; "
                "not physical causality"
            ),
            "mean_js_sar": mean_js_sar,
            "mean_js_optical": mean_js_opt,
            "sar_share": float(
                sar_share
            ),
            "optical_share": float(
                optical_share
            ),
            "dominance": dominance_label(
                sar_share,
                optical_share,
            ),
            "dominance_share_threshold": DOMINANCE_SHARE,
            "mean_fragility": float(
                fragility[coverage].mean()
            ),
            "high_fragility_tile_threshold": HIGH_FRAGILITY_THRESHOLD,
            "high_fragility_tile_fraction": high_fragility_tile_fraction,
            "flip_rate_no_sar": float(
                sm["flip_no_sar"][coverage].mean()
            ),
            "flip_rate_no_optical": float(
                sm["flip_no_optical"][coverage].mean()
            ),
            "temporary_flood_js_sar": masked_mean(
                js_sar,
                temporary,
            ),
            "temporary_flood_js_optical": masked_mean(
                js_opt,
                temporary,
            ),
        },

        "spatial_geometry": {
            "temporary_flood_connected_components_8conn": (
                temp_geom["component_count"]
            ),
            "largest_temporary_component_pixels": (
                temp_geom["largest_component_pixels"]
            ),
            "largest_temporary_component_area_km2": (
                temp_geom[
                    "largest_component_area_km2"
                ]
            ),
            "largest_temporary_component_fraction_of_temp": (
                temp_geom[
                    "largest_component_fraction_of_temp"
                ]
            ),
            "largest_temporary_component_centroid": (
                temp_geom[
                    "largest_component_centroid"
                ]
            ),
        },

        "visual_evidence": {
            "panel_file": (
                f"images/{scene_id}_evidence.png"
            ),
            "contains_ground_truth": False,
        },

        "grounding_policy": {
            "allowed": [
                "describe predicted class extent",
                "describe calibrated uncertainty",
                "describe P(clear) sensor reliability",
                "describe counterfactual SAR/optical model dependence",
                "describe deterministic spatial geometry",
            ],
            "prohibited": [
                "claim physical causality from counterfactual attribution",
                "claim prediction is ground truth",
                "invent flood cause, damage, population impact, or future evolution",
                "replace the segmentation detector with language-model judgement",
            ],
        },
    }

    return packet

def save_evidence_panel(scene_id, sm, out_path):
    cov = (
        sm["coverage"]
        .astype(bool)
    )

    rgb = stretch_rgb(
        sm["rgb_reflectance"],
        cov,
    )

    vv = stretch_channel(
        sm["sar_vv_db"],
        cov,
    )

    frag = np.maximum(
        sm["js_sar"],
        sm["js_optical"],
    )

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(14, 7),
    )

    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title("Optical RGB")

    axes[0, 1].imshow(vv)
    axes[0, 1].set_title("SAR VV")

    axes[0, 2].imshow(
        sm["pclear"],
        vmin=0,
        vmax=1,
    )
    axes[0, 2].set_title("P(clear)")

    axes[0, 3].imshow(
        sm["prediction"],
        vmin=0,
        vmax=2,
    )
    axes[0, 3].set_title(
        "04H-CAL prediction"
    )

    axes[1, 0].imshow(
        sm["entropy"],
        vmin=0,
        vmax=1,
    )
    axes[1, 0].set_title(
        "Predictive entropy"
    )

    axes[1, 1].imshow(
        sm["js_sar"],
        vmin=0,
        vmax=1,
    )
    axes[1, 1].set_title(
        "Counterfactual JS: SAR"
    )

    axes[1, 2].imshow(
        sm["js_optical"],
        vmin=0,
        vmax=1,
    )
    axes[1, 2].set_title(
        "Counterfactual JS: optical"
    )

    axes[1, 3].imshow(
        frag,
        vmin=0,
        vmax=1,
    )
    axes[1, 3].set_title(
        "Counterfactual fragility"
    )

    for ax in axes.ravel():
        ax.axis("off")

    fig.suptitle(
        f"{scene_id} | deterministic FloodShift evidence",
        fontsize=12,
    )

    fig.tight_layout()

    fig.savefig(
        out_path,
        dpi=120,
        bbox_inches="tight",
    )

    plt.close(fig)

print("P17-C11 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C12 — GENERATE 314 TRAINING PACKETS + PANELS
#
# This is the expensive cell.
# It performs frozen full / no-SAR / no-optical 4-way-TTA
# inference for every original training scene.
# ============================================================
train_packets = []
train_summary_rows = []

start_time = time.time()

model.eval()

for scene_idx, (_, scene_row_series) in enumerate(
    train_df.iterrows(),
    start=1,
):
    scene_row = scene_row_series.to_dict()
    scene_id = str(scene_row["scene_id"])

    sm = infer_one_scene(
        scene_row
    )

    packet = build_packet(
        scene_id,
        scene_row,
        sm,
    )

    # Verify packet before persisting.
    errs = verify_packet(packet)

    if errs:
        raise AssertionError(
            f"Packet verification failed for {scene_id}: {errs}"
        )

    canonical = json.dumps(
        packet,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

    packet_sha = hashlib.sha256(
        canonical
    ).hexdigest()

    packet["provenance"][
        "packet_sha256"
    ] = packet_sha

    packet["provenance"][
        "verifier_status"
    ] = "PASS"

    panel_path = (
        TRAIN_PANEL_ROOT
        / f"{scene_id}_evidence.png"
    )

    save_evidence_panel(
        scene_id,
        sm,
        panel_path,
    )

    train_packets.append(
        packet
    )

    train_summary_rows.append({
        "scene_id": scene_id,
        "temporary_flood_fraction": (
            packet["segmentation"][
                "temporary_flood"
            ][
                "fraction_of_coverage"
            ]
        ),
        "temporary_flood_area_km2": (
            packet["segmentation"][
                "temporary_flood"
            ][
                "area_km2"
            ]
        ),
        "mean_entropy": (
            packet["uncertainty"][
                "mean_entropy"
            ]
        ),
        "review10_fraction": (
            packet["uncertainty"][
                "review10_fraction"
            ]
        ),
        "mean_pclear": (
            packet["sensor_quality"][
                "mean_pclear"
            ]
        ),
        "sar_share": (
            packet[
                "counterfactual_attribution"
            ][
                "sar_share"
            ]
        ),
        "optical_share": (
            packet[
                "counterfactual_attribution"
            ][
                "optical_share"
            ]
        ),
        "dominance": (
            packet[
                "counterfactual_attribution"
            ][
                "dominance"
            ]
        ),
    })

    # Free per-scene maps immediately.
    del sm

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if (
        scene_idx == 1
        or scene_idx % 10 == 0
        or scene_idx == len(train_df)
    ):
        elapsed = (
            time.time()
            - start_time
        ) / 60.0

        print(
            f"[{scene_idx:03d}/{len(train_df)}] "
            f"{scene_id} | "
            f"elapsed={elapsed:.1f} min"
        )

assert len(train_packets) == 314
assert all(
    p["provenance"]["verifier_status"] == "PASS"
    for p in train_packets
)

print("=" * 112)
print("P17-C12 — TRAIN PACKETS COMPLETE")
print("=" * 112)
print("Packets          :", len(train_packets))
print("Panels           :", len(list(IMAGE_ROOT.glob('*_evidence.png'))))
print("Verifier failures: 0")
print("Ground truth used: NO")
print("Frozen test used : NO")
print("Minutes          :", round((time.time() - start_time) / 60.0, 2))
print("P17-C12 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C13 — COPY + VERIFY 34 HELD-OUT VALIDATION PANELS
# ============================================================
def recompute_packet_sha(packet):
    packet_copy = json.loads(
        json.dumps(packet)
    )

    prov = packet_copy["provenance"]

    expected = prov.pop(
        "packet_sha256"
    )

    prov.pop(
        "verifier_status",
        None,
    )

    canonical = json.dumps(
        packet_copy,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

    actual = hashlib.sha256(
        canonical
    ).hexdigest()

    return expected, actual

# Find all Notebook-07 panel files.
all_input_panels = list(
    INPUT_ROOT.rglob(
        "*_evidence.png"
    )
)

panel_by_name = {
    p.name: p
    for p in all_input_panels
}

assert len(val_packets_07) == 34

for packet in val_packets_07:
    assert packet["provenance"]["verifier_status"] == "PASS"
    assert packet["visual_evidence"]["contains_ground_truth"] is False

    expected_sha, actual_sha = recompute_packet_sha(
        packet
    )

    assert expected_sha == actual_sha, (
        f"Notebook-07 packet hash mismatch for {packet['scene_id']}"
    )

    scene_id = packet["scene_id"]
    panel_name = f"{scene_id}_evidence.png"

    assert panel_name in panel_by_name, (
        f"Missing validation panel: {panel_name}"
    )

    dst = (
        IMAGE_ROOT
        / panel_name
    )

    if not dst.exists():
        shutil.copy2(
            panel_by_name[panel_name],
            dst,
        )

assert len(
    list(
        IMAGE_ROOT.glob(
            "*_evidence.png"
        )
    )
) == 348

print("=" * 112)
print("P17-C13 — HELD-OUT VALIDATION SOURCE")
print("=" * 112)
print("Validation packets :", len(val_packets_07))
print("Validation panels  :", len(val_packets_07))
print("Packet SHA failures: 0")
print("Total corpus images:", len(list(IMAGE_ROOT.glob('*_evidence.png'))))
print("P17-C13 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C14 — FIXED TEXT / ABSTENTION POLICY
# ============================================================
# These are response-policy thresholds for instruction generation.
# They are NOT segmentation metrics and are not tuned on labels.
SCENE_REVIEW_PIXEL_FRACTION = 0.20
SCENE_FRAGILITY_TILE_FRACTION = 0.50

def pct(x, decimals=2):
    if x is None:
        return "unavailable"
    return f"{100.0 * float(x):.{decimals}f}%"

def dec(x, decimals=3):
    if x is None:
        return "unavailable"
    return f"{float(x):.{decimals}f}"

def area(x):
    if x is None:
        return "unavailable"
    return f"{float(x):.2f} km²"

def integer(x):
    return str(int(x))

def scene_requires_review(packet):
    review10 = float(
        packet["uncertainty"]["review10_fraction"]
    )
    fragile = float(
        packet["counterfactual_attribution"][
            "high_fragility_tile_fraction"
        ]
    )

    return (
        review10 >= SCENE_REVIEW_PIXEL_FRACTION
        or fragile >= SCENE_FRAGILITY_TILE_FRACTION
    )

def example_id(scene_id, task):
    return f"{scene_id}__{task}"

print("=" * 112)
print("P17-C14 — RESPONSE POLICY")
print("=" * 112)
print(
    "Scene review rule:",
    f"review10 >= {SCENE_REVIEW_PIXEL_FRACTION:.2f}",
    "OR",
    f"high-fragility-tile fraction >= {SCENE_FRAGILITY_TILE_FRACTION:.2f}",
)
print("No label-derived thresholding is used.")
print("P17-C14 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C15 — DETERMINISTIC TASK GENERATOR
# ============================================================
TASKS = [
    "extent",
    "uncertainty",
    "sensor_attribution",
    "temporary_sensor_attribution",
    "spatial_geometry",
    "integrated_summary",
    "review_policy",
    "cause_refusal",
    "impact_refusal",
    "forecast_refusal",
]

def make_example(packet, task):
    sid = packet["scene_id"]
    split = scene_split[sid]

    seg = packet["segmentation"]
    unc = packet["uncertainty"]
    sq = packet["sensor_quality"]
    att = packet["counterfactual_attribution"]
    geo = packet["spatial_geometry"]
    prov = packet["provenance"]

    tmp_frac = seg["temporary_flood"]["fraction_of_coverage"]
    tmp_area = seg["temporary_flood"]["area_km2"]
    perm_frac = seg["permanent_water"]["fraction_of_coverage"]

    mean_ent = unc["mean_entropy"]
    review10 = unc["review10_fraction"]
    review20 = unc["review20_fraction"]

    mean_pclear = sq["mean_pclear"]
    low_pclear = sq["low_pclear_fraction"]

    sar_share = att["sar_share"]
    opt_share = att["optical_share"]
    dominance = att["dominance"]
    flip_sar = att["flip_rate_no_sar"]
    flip_opt = att["flip_rate_no_optical"]

    tmp_js_sar = att["temporary_flood_js_sar"]
    tmp_js_opt = att["temporary_flood_js_optical"]

    high_frag = att["high_fragility_tile_fraction"]
    mean_frag = att["mean_fragility"]

    facts = []
    mode = "answer"

    if task == "extent":
        instruction = (
            "Using only the verified FloodShift evidence, summarize the "
            "predicted temporary-flood and permanent-water extent in this scene."
        )

        response = (
            f"FloodShift predicts temporary flooding over {pct(tmp_frac)} "
            f"of the sensor-covered scene"
        )

        if tmp_area is not None:
            response += f", corresponding to {area(tmp_area)}"

        response += (
            f". Permanent water accounts for {pct(perm_frac)} of the "
            "sensor-covered scene. These values are model-derived predictions, "
            "not ground truth."
        )

        facts = [
            "segmentation.temporary_flood.fraction_of_coverage",
            "segmentation.temporary_flood.area_km2",
            "segmentation.permanent_water.fraction_of_coverage",
        ]

    elif task == "uncertainty":
        instruction = (
            "How uncertain is the FloodShift prediction for this scene? "
            "Use only the calibrated uncertainty facts."
        )

        response = (
            f"The mean normalized predictive entropy is {dec(mean_ent)}. "
            f"{pct(review10)} of sensor-covered pixels exceed the frozen "
            f"review-10 threshold, and {pct(review20)} exceed the frozen "
            f"review-20 threshold. Mean optical reliability P(clear) is "
            f"{dec(mean_pclear)}, with {pct(low_pclear)} of covered pixels "
            f"below P(clear)=0.50."
        )

        facts = [
            "uncertainty.mean_entropy",
            "uncertainty.review10_fraction",
            "uncertainty.review20_fraction",
            "sensor_quality.mean_pclear",
            "sensor_quality.low_pclear_fraction",
        ]

    elif task == "sensor_attribution":
        instruction = (
            "Which sensor does the current FloodShift prediction depend on "
            "more strongly, and what does that attribution mean?"
        )

        response = (
            f"The counterfactual attribution is {dominance}. "
            f"Optical contributes {pct(opt_share)} of the normalized "
            f"SAR-versus-optical attribution share and SAR contributes "
            f"{pct(sar_share)}. Removing optical changes the predicted class "
            f"for {pct(flip_opt)} of covered pixels, while removing SAR changes "
            f"{pct(flip_sar)}. These scores measure model dependence under "
            f"counterfactual sensor removal; they do not establish physical causality."
        )

        facts = [
            "counterfactual_attribution.dominance",
            "counterfactual_attribution.optical_share",
            "counterfactual_attribution.sar_share",
            "counterfactual_attribution.flip_rate_no_optical",
            "counterfactual_attribution.flip_rate_no_sar",
        ]

    elif task == "temporary_sensor_attribution":
        instruction = (
            "For pixels predicted as temporary flood, compare FloodShift's "
            "counterfactual sensitivity to SAR removal and optical removal."
        )

        if tmp_js_sar is None or tmp_js_opt is None:
            response = (
                "This evidence packet does not contain a valid temporary-flood "
                "counterfactual attribution value for this scene, so a "
                "temporary-flood-specific sensor comparison is not supported."
            )
            mode = "abstain"
        else:
            dominant_tmp = (
                "optical"
                if float(tmp_js_opt) > float(tmp_js_sar)
                else "SAR"
            )

            response = (
                f"Within pixels predicted as temporary flood, normalized "
                f"counterfactual JS sensitivity is {dec(tmp_js_opt)} for optical "
                f"removal and {dec(tmp_js_sar)} for SAR removal. The temporary-"
                f"flood prediction is therefore more sensitive to {dominant_tmp} "
                f"removal in this scene. This is model-dependence evidence, "
                f"not physical causality."
            )

        facts = [
            "counterfactual_attribution.temporary_flood_js_optical",
            "counterfactual_attribution.temporary_flood_js_sar",
        ]

    elif task == "spatial_geometry":
        instruction = (
            "Describe the deterministic spatial geometry of the predicted "
            "temporary-flood mask without interpreting connected components "
            "as separate flood events."
        )

        count = geo[
            "temporary_flood_connected_components_8conn"
        ]

        largest_px = geo[
            "largest_temporary_component_pixels"
        ]

        largest_area = geo[
            "largest_temporary_component_area_km2"
        ]

        response = (
            f"The predicted temporary-flood mask contains {integer(count)} "
            f"pixel-level 8-connected components. The largest component contains "
            f"{integer(largest_px)} pixels"
        )

        if largest_area is not None:
            response += f" and covers {area(largest_area)}"

        response += (
            ". These connected components are image-mask structures and must "
            "not be interpreted as distinct real-world flood events."
        )

        facts = [
            "spatial_geometry.temporary_flood_connected_components_8conn",
            "spatial_geometry.largest_temporary_component_pixels",
            "spatial_geometry.largest_temporary_component_area_km2",
        ]

    elif task == "integrated_summary":
        instruction = (
            "Give a concise scientific summary of the verified FloodShift "
            "evidence for this scene. Include extent, uncertainty, optical "
            "reliability, and sensor dependence."
        )

        response = (
            f"FloodShift predicts temporary flooding over {pct(tmp_frac)} "
            f"of the sensor-covered scene"
        )

        if tmp_area is not None:
            response += f" ({area(tmp_area)})"

        response += (
            f". Mean normalized predictive entropy is {dec(mean_ent)}, and "
            f"{pct(review10)} of covered pixels exceed the frozen review-10 "
            f"threshold. Mean P(clear) is {dec(mean_pclear)}. Counterfactual "
            f"sensor attribution is {dominance}, with optical share "
            f"{pct(opt_share)} and SAR share {pct(sar_share)}. The attribution "
            f"describes model dependence, not physical causality."
        )

        facts = [
            "segmentation.temporary_flood.fraction_of_coverage",
            "segmentation.temporary_flood.area_km2",
            "uncertainty.mean_entropy",
            "uncertainty.review10_fraction",
            "sensor_quality.mean_pclear",
            "counterfactual_attribution.dominance",
            "counterfactual_attribution.optical_share",
            "counterfactual_attribution.sar_share",
        ]

    elif task == "review_policy":
        instruction = (
            "Should this scene be flagged for additional human review under "
            "the fixed FloodShift instruction-dataset review policy?"
        )

        needs_review = scene_requires_review(packet)

        if needs_review:
            response = (
                f"Yes. Under the fixed scene-level review policy, this scene is "
                f"flagged because {pct(review10)} of covered pixels exceed the "
                f"frozen review-10 entropy threshold and {pct(high_frag)} of its "
                f"tiles exceed the frozen counterfactual-fragility threshold. "
                f"This flag requests review; it does not declare the prediction incorrect."
            )
            mode = "abstain"
        else:
            response = (
                f"No additional scene-level flag is triggered by the fixed policy: "
                f"{pct(review10)} of covered pixels exceed the review-10 entropy "
                f"threshold and {pct(high_frag)} of tiles exceed the frozen "
                f"counterfactual-fragility threshold. This does not make the "
                f"prediction ground truth."
            )

        facts = [
            "uncertainty.review10_fraction",
            "counterfactual_attribution.high_fragility_tile_fraction",
        ]

    elif task == "cause_refusal":
        instruction = (
            "What caused the flooding in this scene? Identify the physical cause."
        )

        response = (
            "The verified FloodShift evidence does not establish the physical "
            "cause of flooding. Counterfactual SAR/optical attribution measures "
            "the model's dependence on sensor inputs, not environmental causality. "
            "A physical-cause claim would require external hydrologic, meteorological, "
            "or event data that are not present in this packet."
        )

        facts = []
        mode = "refuse"

    elif task == "impact_refusal":
        instruction = (
            "How many people or buildings were affected by the predicted flooding?"
        )

        response = (
            "The FloodShift evidence packet does not contain population, building, "
            "damage, or exposure data. It supports predicted flood extent and "
            "model-reliability facts only, so the number of affected people or "
            "buildings cannot be determined from this evidence."
        )

        facts = []
        mode = "refuse"

    elif task == "forecast_refusal":
        instruction = (
            "Will flooding in this scene become worse tomorrow?"
        )

        response = (
            "This static FloodShift evidence packet does not provide a future "
            "forecast. It describes the current model prediction, uncertainty, "
            "sensor reliability, and counterfactual sensor dependence. Predicting "
            "tomorrow's flood evolution would require temporal or forecast inputs "
            "that are not contained in this evidence."
        )

        facts = []
        mode = "refuse"

    else:
        raise ValueError(task)

    return {
        "id": example_id(sid, task),
        "scene_id": sid,
        "split": split,
        "task": task,
        "image": f"images/{sid}_evidence.png",
        "instruction": instruction,
        "response": response,
        "response_mode": mode,
        "grounded_fact_paths": facts,
        "source_packet_sha256": prov["packet_sha256"],
        "source_segmenter": prov["segmenter"],
        "source_test_status": prov["test_status"],
    }

print("P17-C15 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C16 — FINAL SPLIT + 3,480 INSTRUCTION EXAMPLES
# ============================================================
packets = (
    train_packets
    + val_packets_07
)

assert len(packets) == 348

train_scene_ids = {
    p["scene_id"]
    for p in train_packets
}

validation_scene_ids = {
    p["scene_id"]
    for p in val_packets_07
}

assert len(train_scene_ids) == 314
assert len(validation_scene_ids) == 34
assert train_scene_ids.isdisjoint(
    validation_scene_ids
)

scene_split = {
    sid: "train"
    for sid in train_scene_ids
}

scene_split.update({
    sid: "validation"
    for sid in validation_scene_ids
})

examples = []

for packet in packets:
    for task in TASKS:
        examples.append(
            make_example(
                packet,
                task,
            )
        )

train_examples = [
    x
    for x in examples
    if x["split"] == "train"
]

val_examples = [
    x
    for x in examples
    if x["split"] == "validation"
]

assert len(examples) == 3480
assert len(train_examples) == 3140
assert len(val_examples) == 340

assert len({
    x["id"]
    for x in examples
}) == 3480

print("=" * 112)
print("P17-C16 — FINAL CORPUS GENERATED")
print("=" * 112)
print("Total scenes        :", len(packets))
print("Training scenes     :", len(train_scene_ids))
print("Validation scenes   :", len(validation_scene_ids))
print("Scene overlap       :", len(train_scene_ids & validation_scene_ids))
print("Training examples   :", len(train_examples))
print("Validation examples :", len(val_examples))
print("Total examples      :", len(examples))
print("Frozen test         : untouched")
print("P17-C16 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C17 — GROUNDING / SAFETY VERIFIER
# ============================================================
packet_by_scene = {
    p["scene_id"]: p
    for p in packets
}

def resolve_path(obj, path):
    cur = obj

    for token in path.split("."):
        cur = cur[token]

    return cur

def canonical_fact_render(path, value):
    if value is None:
        return None

    if (
        path.endswith("fraction_of_coverage")
        or path.endswith("review10_fraction")
        or path.endswith("review20_fraction")
        or path.endswith("low_pclear_fraction")
        or path.endswith("sar_share")
        or path.endswith("optical_share")
        or path.endswith("flip_rate_no_sar")
        or path.endswith("flip_rate_no_optical")
        or path.endswith("high_fragility_tile_fraction")
    ):
        return pct(value)

    if path.endswith("area_km2"):
        return area(value)

    if (
        path.endswith("mean_entropy")
        or path.endswith("mean_pclear")
        or path.endswith("temporary_flood_js_sar")
        or path.endswith("temporary_flood_js_optical")
    ):
        return dec(value)

    if (
        path.endswith("connected_components_8conn")
        or path.endswith("largest_temporary_component_pixels")
    ):
        return integer(value)

    if path.endswith("dominance"):
        return str(value)

    return str(value)

verification_rows = []

for ex in examples:
    packet = packet_by_scene[
        ex["scene_id"]
    ]

    errors = []

    if (
        ex["source_packet_sha256"]
        != packet["provenance"]["packet_sha256"]
    ):
        errors.append(
            "source packet SHA mismatch"
        )

    if (
        ex["source_segmenter"]
        != "FloodShift-04H-CAL"
    ):
        errors.append(
            "wrong segmenter"
        )

    if (
        ex["source_test_status"]
        != "LOCKED_NOT_EVALUATED"
    ):
        errors.append(
            "test status changed"
        )

    if ex["response_mode"] == "answer":
        for path in ex["grounded_fact_paths"]:
            value = resolve_path(
                packet,
                path,
            )

            rendered = canonical_fact_render(
                path,
                value,
            )

            if (
                rendered is not None
                and rendered not in ex["response"]
            ):
                errors.append(
                    f"rendered fact not present: {path} -> {rendered}"
                )

    if ex["response_mode"] == "refuse":
        if re.search(
            r"\d+\.\d+%|\d+\.\d+\s*km²",
            ex["response"],
        ):
            errors.append(
                "refusal response contains unsupported numeric claim"
            )

    banned_response_phrases = [
        "definitely caused by",
        "proves that",
        "ground truth confirms",
        "will definitely",
        "people were affected",
        "buildings were damaged",
    ]

    response_lower = (
        ex["response"]
        .lower()
    )

    for phrase in banned_response_phrases:
        if phrase in response_lower:
            errors.append(
                f"banned phrase: {phrase}"
            )

    verification_rows.append({
        "id": ex["id"],
        "scene_id": ex["scene_id"],
        "task": ex["task"],
        "response_mode": ex["response_mode"],
        "errors": errors,
        "pass": len(errors) == 0,
    })

verification_df = pd.DataFrame(
    verification_rows
)

failed = verification_df[
    ~verification_df["pass"]
]

if len(failed):
    display(failed.head(30))
    raise AssertionError(
        f"{len(failed)} examples failed grounding verification."
    )

print("=" * 112)
print("P17-C17 — GROUNDING VERIFIER")
print("=" * 112)
print("Verified examples :", len(verification_df))
print("Failures          :", len(failed))
print("Training scenes   :", len(train_scene_ids))
print("Validation scenes :", len(validation_scene_ids))
print("Scene leakage     :", len(train_scene_ids & validation_scene_ids))
print("Frozen test       : untouched")
print("P17-C17 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C18 — CORPUS STATISTICS
# ============================================================
examples_df = pd.DataFrame([
    {
        "id": x["id"],
        "scene_id": x["scene_id"],
        "split": x["split"],
        "task": x["task"],
        "response_mode": x["response_mode"],
        "response_chars": len(x["response"]),
    }
    for x in examples
])

task_stats = (
    examples_df
    .groupby(
        [
            "split",
            "task",
            "response_mode",
        ],
        as_index=False,
    )
    .agg(
        examples=("id", "count"),
        scenes=("scene_id", "nunique"),
        mean_response_chars=("response_chars", "mean"),
    )
)

mode_stats = (
    examples_df
    .groupby(
        [
            "split",
            "response_mode",
        ],
        as_index=False,
    )
    .agg(
        examples=("id", "count")
    )
)

train_packet_df = pd.DataFrame(
    train_summary_rows
)

print("=" * 112)
print("P17-C18 — FULL CORPUS AUDIT")
print("=" * 112)
print("Response-mode distribution:")
display(mode_stats)

print("Task balance:")
display(task_stats)

print("Training-packet sensor dominance:")
display(
    train_packet_df[
        "dominance"
    ]
    .value_counts()
    .rename_axis("dominance")
    .reset_index(name="scenes")
)

print(
    "Training mean temporary-flood fraction:",
    float(
        train_packet_df[
            "temporary_flood_fraction"
        ].mean()
    ),
)

print(
    "Training mean entropy:",
    float(
        train_packet_df[
            "mean_entropy"
        ].mean()
    ),
)

print(
    "Training mean optical share:",
    float(
        train_packet_df[
            "optical_share"
        ].mean()
    ),
)

print("P17-C18 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C19 — EXPORT FINAL VLM CORPUS
# ============================================================
TRAIN_JSONL = (
    DATASET_ROOT
    / "FloodShift_08B_train.jsonl"
)

VAL_JSONL = (
    DATASET_ROOT
    / "FloodShift_08B_validation.jsonl"
)

ALL_JSONL = (
    DATASET_ROOT
    / "FloodShift_08B_all.jsonl"
)

LLAVA_TRAIN_JSON = (
    DATASET_ROOT
    / "FloodShift_08B_llava_train.json"
)

LLAVA_VAL_JSON = (
    DATASET_ROOT
    / "FloodShift_08B_llava_validation.json"
)

TRAIN_PACKETS_JSONL = (
    DATASET_ROOT
    / "FloodShift_08B_train_scene_packets.jsonl"
)

VAL_PACKETS_JSONL = (
    DATASET_ROOT
    / "FloodShift_08B_validation_scene_packets.jsonl"
)

VERIFICATION_CSV = (
    DATASET_ROOT
    / "FloodShift_08B_grounding_verification.csv"
)

TASK_STATS_CSV = (
    DATASET_ROOT
    / "FloodShift_08B_task_stats.csv"
)

TRAIN_SUMMARY_CSV = (
    DATASET_ROOT
    / "FloodShift_08B_train_scene_summary.csv"
)

DATASET_CARD = (
    DATASET_ROOT
    / "FloodShift_08B_dataset_card.json"
)

PROTOCOL_PATH = (
    OUTPUT_ROOT
    / "FloodShift_08B_protocol_record.json"
)

def write_jsonl(path, rows):
    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:
        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                    sort_keys=True,
                    allow_nan=False,
                )
                + "\n"
            )

write_jsonl(
    TRAIN_JSONL,
    train_examples,
)

write_jsonl(
    VAL_JSONL,
    val_examples,
)

write_jsonl(
    ALL_JSONL,
    examples,
)

write_jsonl(
    TRAIN_PACKETS_JSONL,
    train_packets,
)

write_jsonl(
    VAL_PACKETS_JSONL,
    val_packets_07,
)

def to_llava(ex):
    return {
        "id": ex["id"],
        "image": ex["image"],
        "conversations": [
            {
                "from": "human",
                "value": (
                    "<image>\n"
                    + ex["instruction"]
                ),
            },
            {
                "from": "gpt",
                "value": ex["response"],
            },
        ],
        "metadata": {
            "scene_id": ex["scene_id"],
            "split": ex["split"],
            "task": ex["task"],
            "response_mode": ex["response_mode"],
            "grounded_fact_paths": ex["grounded_fact_paths"],
            "source_packet_sha256": ex["source_packet_sha256"],
        },
    }

with open(
    LLAVA_TRAIN_JSON,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        [
            to_llava(x)
            for x in train_examples
        ],
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(
    LLAVA_VAL_JSON,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        [
            to_llava(x)
            for x in val_examples
        ],
        f,
        ensure_ascii=False,
        indent=2,
    )

verification_df.to_csv(
    VERIFICATION_CSV,
    index=False,
)

task_stats.to_csv(
    TASK_STATS_CSV,
    index=False,
)

train_packet_df.to_csv(
    TRAIN_SUMMARY_CSV,
    index=False,
)

dataset_card = {
    "name": "FloodShift-08B-Full-Grounded-Multimodal-Corpus",
    "version": "1.0",
    "source_segmenter": "FloodShift-04H-CAL",
    "source_04h_sha256": SHA_04H,
    "source_07_archive_sha256": ARCHIVE_07_SHA,
    "training_scenes": 314,
    "validation_scenes": 34,
    "frozen_geographic_test_scenes": 58,
    "training_examples": 3140,
    "validation_examples": 340,
    "total_examples": 3480,
    "examples_per_scene": len(TASKS),
    "scene_overlap_train_validation": 0,
    "ground_truth_in_language_packets": False,
    "external_llm_targets_used": False,
    "deterministic_template_targets": True,
    "grounding_verification_failures": 0,
    "test_status": "LOCKED_NOT_EVALUATED",
}

with open(
    DATASET_CARD,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        dataset_card,
        f,
        indent=2,
    )

protocol08b = {
    "experiment": "FloodShift-08B-Full-Grounded-Corpus",
    "source_segmenter": "FloodShift-04H-CAL",
    "source_04h_sha256": SHA_04H,
    "source_uncertainty_stage": "FloodShift-05",
    "source_attribution_stage": "FloodShift-06",
    "validation_source_stage": "FloodShift-07",
    "05b_status": "NOT_PROMOTED_NOT_USED",
    "training_scenes": 314,
    "validation_scenes": 34,
    "training_examples": 3140,
    "validation_examples": 340,
    "scene_overlap": 0,
    "grounding_verification_failures": 0,
    "ground_truth_in_language_packets": False,
    "external_llm_generation_used": False,
    "test_status": "LOCKED_NOT_EVALUATED",
    "next_stage": "Gaussian_latent_variational_multimodal_language_model",
}

with open(
    PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        protocol08b,
        f,
        indent=2,
    )

print("=" * 112)
print("P17-C19 — FINAL CORPUS EXPORT")
print("=" * 112)
print("Train JSONL      :", TRAIN_JSONL)
print("Validation JSONL :", VAL_JSONL)
print("LLaVA train      :", LLAVA_TRAIN_JSON)
print("LLaVA validation :", LLAVA_VAL_JSON)
print("Images           :", len(list(IMAGE_ROOT.glob('*_evidence.png'))))
print("Dataset card     :", DATASET_CARD)
print("Protocol         :", PROTOCOL_PATH)
print("P17-C19 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P17-C20 — PER-FILE MANIFEST + FINAL ARCHIVE
# ============================================================
MANIFEST_PATH = (
    OUTPUT_ROOT
    / "FloodShift_08B_file_manifest.csv"
)

manifest_rows = []

for p in sorted(
    DATASET_ROOT.rglob("*")
):
    if p.is_file():
        manifest_rows.append({
            "relative_path": str(
                p.relative_to(
                    OUTPUT_ROOT
                )
            ),
            "bytes": int(
                p.stat().st_size
            ),
            "sha256": sha256_file(
                p
            ),
        })

manifest_df = pd.DataFrame(
    manifest_rows
)

manifest_df.to_csv(
    MANIFEST_PATH,
    index=False,
)

ARCHIVE_PATH = (
    OUTPUT_ROOT
    / "FloodShift_08B_ARCHIVE.zip"
)

with zipfile.ZipFile(
    ARCHIVE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for p in sorted(
        DATASET_ROOT.rglob("*")
    ):
        if p.is_file():
            zf.write(
                p,
                arcname=p.relative_to(
                    OUTPUT_ROOT
                ),
            )

    zf.write(
        PROTOCOL_PATH,
        arcname=PROTOCOL_PATH.relative_to(
            OUTPUT_ROOT
        ),
    )

    zf.write(
        MANIFEST_PATH,
        arcname=MANIFEST_PATH.relative_to(
            OUTPUT_ROOT
        ),
    )

ARCHIVE_SHA = sha256_file(
    ARCHIVE_PATH
)

print("=" * 112)
print("P17-C20 — FINAL ARCHIVE")
print("=" * 112)
print("Training scenes      :", len(train_scene_ids))
print("Validation scenes    :", len(validation_scene_ids))
print("Training examples    :", len(train_examples))
print("Validation examples  :", len(val_examples))
print("Images               :", len(list(IMAGE_ROOT.glob('*_evidence.png'))))
print("Grounding failures   :", len(failed))
print("Scene leakage        :", len(train_scene_ids & validation_scene_ids))
print("Ground truth packets : False")
print("Frozen geographic test: LOCKED_NOT_EVALUATED")
print("File manifest rows   :", len(manifest_df))
print("Archive              :", ARCHIVE_PATH)
print("Archive SHA256       :", ARCHIVE_SHA)
print()
print("NEXT: Gaussian-latent variational multimodal language model.")
print("P17-C20 COMPLETE ✅")


## After the run

The expensive stage is **P17-C12** because it performs frozen three-path, four-way-TTA inference over all 314 training scenes.

After completion, send me the executed notebook or the outputs from:

- **P17-C12** — 314 training packet generation
- **P17-C13** — held-out validation verification
- **P17-C16** — final scene/example counts
- **P17-C17** — grounding verifier
- **P17-C18** — corpus statistics
- **P17-C20** — final archive SHA256

Download:

```text
FloodShift_08B_ARCHIVE.zip
```

This is the actual corpus we should use for the next stage.

Expected final structure:

```text
314 train scenes
34 validation scenes
0 scene overlap
58 geographic test scenes still locked

3140 train instructions
340 validation instructions
3480 total examples
348 evidence panels
```

The next notebook will implement the **Gaussian-latent variational multimodal language model** with:

\[
\mu,\log\sigma^2
\rightarrow
z=\mu+\sigma\odot\epsilon
\]

and will train only on this expanded grounded corpus.
